# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
meta = dataset.metadata

print(f"Dataset Title: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}\nPublished: {getattr(meta, 'datePublished', None)}\n")
print(f"Cite as: {getattr(meta, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields (columns) they contain. All items are referenced by their `@id`.

In [ ]:
from mlcroissant._src.structs.metadata import RecordSet

# Display all available record sets and their fields
record_sets = []
print("Available record sets:")
for rs in dataset.metadata.record_sets:
    record_sets.append(rs['@id'])
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', None)}")
    print(f"  Description: {rs.get('description', None)}")
    print(f"  Field @ids:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']} (name: {field.get('name', None)})")
    print()

## Preview Record Structure
You can print records from a record set by `@id`. Here we print the first 2 records for each set.

In [ ]:
# Print a preview of records for each record set
# (edit 'record_sets' below to reference your desired sets)
for record_set_id in record_sets:
    print(f"\n--- RecordSet: {record_set_id} ---")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i == 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract all records from all record sets to pandas DataFrames by @id
dfs = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)

# Show available dataframes and their columns
for rsid, df in dfs.items():
    print(f"\nRecordSet: {rsid}")
    print("Columns:", df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we'll pick a likely protein quantitative field (such as `cr:Abundance` or similar, by `@id`) and a categorical/class/grouping field such as `cr:Description`.

In [ ]:
# You may need to change these IDs below based on the record set/column list outputs above.
# Example IDs for protein abundance and description group field (replace as needed):
example_record_set = record_sets[0]  # pick the main protein table as default

# View the head to choose field ids
print(f"Available fields in {example_record_set}: {dfs[example_record_set].columns.tolist()}")

# Try to smartly guess numeric and group fields
possible_numeric = [c for c in dfs[example_record_set].columns if 'bund' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower() or 'count' in c.lower() or 'abundance' in c.lower()]
possible_group = [c for c in dfs[example_record_set].columns if 'desc' in c.lower() or 'name' in c.lower() or 'group' in c.lower() or 'sample' in c.lower()]
print(f"Possible numeric fields: {possible_numeric}")
print(f"Possible group fields: {possible_group}")

# Assign or fallback to available
numeric_field_id = possible_numeric[0] if possible_numeric else dfs[example_record_set].columns[0]
group_field_id = possible_group[0] if possible_group else dfs[example_record_set].columns[0]

# Coerce to float for numeric analysis
df = dfs[example_record_set].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fc' else 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the main quantitative field and group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for numeric field
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If group means exist, show top 10
if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    grouped_df.nlargest(10).plot(kind='bar')
    plt.title(f'Top 10 {group_field_id} mean {numeric_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant-packaged FAIR^2 dataset using the `mlcroissant` library. We discovered record sets, referenced fields using their `@id`, extracted the data into pandas, performed basic normalization and grouping, and visualized key statistics and distributions. This process supports FAIR data practices and reproducible research for mass spectrometry datasets and beyond.